# 03 — EDA (Análisis Exploratorio de Datos)

**Objetivo:** entender el negocio con datos para tomar decisiones informadas sobre el modelado.

El propósito de este notebook es responder preguntas concretas sobre el comportamiento del negocio:

- ¿Cuál es el patrón de ventas diarias? ¿Hay estacionalidad semanal, mensual, por temporada?
- ¿Qué variables correlacionan con las ventas? ¿Temperatura, festivos, día de la semana?
- ¿Cómo se comportan las reservas respecto a los tickets reales?
- ¿Qué artículos concentran el 80% de la facturación?
- ¿Hay diferencias relevantes entre turno de comida y cena?

Las respuestas a estas preguntas definen qué features entran al modelo y con qué forma.


*Arquitectura Medallion: Bronze → Silver → **Gold***

### 0. Setup
Imports, rutas, carga de los parquets de Silver. Una sola celda de configuración compartida.

### 1. Visión general de los datos Silver
Tabla resumen de todos los datasets: filas, columnas, cobertura temporal. El equivalente al `audit_dataset` de la depuración, pero aquí como punto de partida descriptivo, no como diagnóstico de calidad (eso ya está hecho).

### 2. Análisis de la variable objetivo: ingresos diarios
La serie temporal de facturación agregada por día es el núcleo del TFM. Todo lo demás es contexto explicativo de esta variable.

- Evolución temporal de facturación diaria (tickets + ventas)
- Distribución del importe por ticket
- Detección visual de outliers y comportamientos anómalos
- Patrón por día de la semana y por turno

### 3. Análisis de reservas
- Volumen diario de reservas por turno y estado (confirmada, cancelada, no-show)
- Tasa de cancelación y no-show en el tiempo
- Relación reservas → comensales reales (si está disponible)
- Antelación de la reserva (creación vs. servicio)

### 4. Análisis de producto (ventas por artículo y departamento)
- Artículos y departamentos más vendidos (volumen e importe)
- Concentración: ¿cuántos artículos explican el 80% de la facturación? (Pareto)
- Estacionalidad por categoría/departamento

> ⚠️ Ojo: `total_articles` es un agregado global sin granularidad diaria. No mezclar con `ventas` sin aclarar qué se está mirando.

### 5. Análisis de propinas (tips)
- Distribución del importe de propina
- Relación propina / importe ticket (ratio)
- Variación por turno o día de la semana si está disponible vía `document_id`

### 6. Variables externas: meteorología
- Temperatura y precipitación en el período de estudio
- Correlación visual (scatter / heatmap) entre variables meteo y facturación diaria
- ¿Hay efecto diferenciado entre días de lluvia y días secos?

### 7. Variables externas: festivos y eventos
- Distribución de facturación en días festivos vs. no festivos
- Comportamiento en fechas de eventos de alto impacto (Pozuelo, La Roca)
- Comparativa visual de medias: festivo vs. laborable, evento vs. sin evento

### 8. Análisis cruzado: construcción de la tabla maestra diaria
Este paso es crítico porque es el que alimenta directamente el modelado. Se trata de unir en una sola tabla por día:

- Facturación diaria (de tickets/ventas)
- N.º reservas y comensales
- Variables meteo diarias
- Flag festivo / nombre festivo
- Flag evento / intensidad sugerida

Comprobar que no hay huecos temporales inesperados tras el cruce.

### 9. Correlaciones y selección preliminar de features
- Matriz de correlación entre facturación diaria y todas las variables numéricas disponibles
- Identificación de las variables con mayor correlación con la variable objetivo
- Primera reflexión sobre qué features tienen sentido para el modelo (justificada, no automática)

### 10. Conclusiones del EDA
Resumen breve y accionable: qué patrones se han encontrado, qué features parecen relevantes, qué limitaciones tiene el dato (cobertura temporal desde oct-2025, tamaño de muestra, ausencia de datos de personal/costes, etc.). Este apartado alimenta directamente la sección del documento final.